# Sprint 4 - Tool Calling and AI Agent foundations

## Business Requirement

Our AI Assistant can retrieve information fro the Employee handbook. It must now use external capailities to perform tasks and access business systems.

#### This note book introdces Tool Calling -> Multiple Tools -> Agent Loop -> Reasoning and Pleanning with Langchain.

## 1.Install Required Libraries

In [ ]:
!pip -q install openai

## 2.Configure OpenAI API

In [2]:
import os
from getpass import getpass
from openai import OpenAI

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"]=getpass("Enter your OpenAI API Key:")

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

GENERATION_MODEL = "gpt-4.1-mini"

print("OpenAI client Configured successfully.")
print("Model:",GENERATION_MODEL)

Enter your OpenAI API Key: ········


OpenAI client Configured successfully.
Model: gpt-4.1-mini


## 3.First Tool - A Python Function

A tool starts with a function that performs a specific task.

In [4]:
def calculate_leave_balance(total_leaves,leaves_taken):
    return {"remaining_leaves":total_leaves - leaves_taken}

result=calculate_leave_balance(24,7)
print(result)

{'remaining_leaves': 17}


## 4.Function Calling - Important Notes

1. The function exists in our application,but the LLM doesn't execute the Python code.
2. The model receives the description of the available function and can return a tool with the required arguments.
3. The application then executes the function and sends the result back to the modl.

## 4.1 Define the Model

In [6]:
tools = [
    {
        "type": "function",
        "name": "calculate_leave_balance",
        "description": "Calculate an employee's remaining leave balance.",
        "parameters": {
            "type": "object",
            "properties": {
                "total_leaves": {
                    "type": "integer",
                    "description": "Total annual leave allowance."
                },
                "leaves_taken": {
                    "type": "integer",
                    "description": "Number of leaves already taken."
                }
            },
            "required": ["total_leaves", "leaves_taken"],
            "additionalProperties": False
        }
    }
]

## 4.2 Send the user's request to the Model

In [7]:
question = "An employee has 24 total leaves and has taken 7.What is the remaining leave balance? "

response = client.responses.create(model=GENERATION_MODEL,input=question,tools=tools)

response.output

[ResponseFunctionToolCall(arguments='{"total_leaves":24,"leaves_taken":7}', call_id='call_EDE7Z54knM3jkaAWp3oIEdBF', name='calculate_leave_balance', type='function_call', id='fc_05242c799e17fdbc006aad6ea970bc87d18917680dec72b99d', async_=None, caller=None, namespace=None, status='completed')]

## 4.3 Execute the Requested Tool

In [10]:
import json

for tool_call in response.output:
    arguments=json.loads(tool_call.arguments)
    result=calculate_leave_balance(**arguments)

    print("==="*30)
    print("Tool:",tool_call.name)
    print("Arguments:",arguments)
    print("Result:",result)

Tool: calculate_leave_balance
Arguments: {'total_leaves': 24, 'leaves_taken': 7}
Result: {'remaining_leaves': 17}


## 5.Complete Tool Calling Workflow

this is the final part of the tool-calling loop. Executed the Python function earlier; now giving its result back to the model so it can produce the final human-readable answer.

In [17]:
import json

tool_outputs = []

for tool_call in response.output:
    if tool_call.type == "function_call":
        arguments = json.loads(tool_call.arguments)

        result = calculate_leave_balance(**arguments)

        tool_outputs.append({
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": json.dumps(result)
        })

final_response = client.responses.create(
    model=GENERATION_MODEL,
    previous_response_id=response.id,
    input=tool_outputs,
    tools=tools
)

print(final_response.output_text)

The employee's remaining leave balance is 17 days.
